In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json


# First load the dataset

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("Cornell-University/arxiv")

print("Path to dataset files:", path)


Path to dataset files: /kaggle/input/arxiv


In [3]:
import pandas as pd

print("[PRINT] Loading JSON file with pandas")

try:
    df_original = pd.read_json('/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json', lines=True)
    print(f"[PRINT] DataFrame created with shape: {df_original.shape}")
except Exception as e:
    print(f"[PRINT] Error loading JSON file: {e}")
    raise


[PRINT] Loading JSON file with pandas
[PRINT] Error loading JSON file: name 'df' is not defined


NameError: name 'df' is not defined

In [4]:
df = df_original.copy()

In [5]:
df

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]"
3,0704.0004,David Callan,David Callan,A determinant of Stirling cycle numbers counts...,11 pages,None,None,None,math.CO,None,We show that a determinant of Stirling cycle...,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2007-05-23,"[[Callan, David, ]]"
4,0704.0005,Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,None,"Illinois J. Math. 52 (2008) no.2, 681-689",None,None,math.CA math.FA,None,In this paper we show how to compute the $\L...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2013-10-15,"[[Abu-Shammala, Wael, ], [Torchinsky, Alberto, ]]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2854534,supr-con/9608008,Ruslan Prozorov,"R. Prozorov, M. Konczykowski, B. Schmidt, Y. Y...",On the origin of the irreversibility line in t...,"19 pages, LaTex, 6 PostScript figures; Author'...",None,10.1103/PhysRevB.54.15530,None,supr-con cond-mat.supr-con,None,We report on measurements of the angular dep...,"[{'version': 'v1', 'created': 'Mon, 26 Aug 199...",2009-10-30,"[[Prozorov, R., ], [Konczykowski, M., ], [Schm..."
2854535,supr-con/9609001,Durga P. Choudhury,"Durga P. Choudhury, Balam A. Willemsen, John S...",Nonlinear Response of HTSC Thin Film Microwave...,"4 pages, LaTeX type, Uses IEEE style files, 60...",None,10.1109/77.620744,None,supr-con cond-mat.supr-con,None,The non-linear microwave surface impedance o...,"[{'version': 'v1', 'created': 'Sat, 31 Aug 199...",2016-11-18,"[[Choudhury, Durga P., , Physics Department, N..."
2854536,supr-con/9609002,Durga P. Choudhury,"Balam A. Willemsen, J. S. Derov and S.Sridhar ...",Critical State Flux Penetration and Linear Mic...,"20 pages, LaTeX type, Uses REVTeX style files,...",None,10.1103/PhysRevB.56.11989,None,supr-con cond-mat.supr-con,None,The vortex contribution to the dc field (H) ...,"[{'version': 'v1', 'created': 'Tue, 3 Sep 1996...",2009-10-30,"[[Willemsen, Balam A., , Physics Department,\n..."
2854537,supr-con/9609003,Hasegawa Yasumasa,Yasumasa Hasegawa (Himeji Institute of Technol...,Density of States and NMR Relaxation Rate in A...,"7 pages, 4 PostScript Figures, LaTeX, to appea...",None,10.1143/JPSJ.65.3131,None,supr-con cond-mat.supr-con,None,We show that the density of states in an ani...,"[{'version': 'v1', 'created': 'Wed, 18 Sep 199...",2009-10-30,"[[Hasegawa, Yasumasa, , Himeji Institute of Te..."


# Preprocessing

## First we need to filter categories to get papers realted to neural network architectures,

In [6]:
unique_categories = df['categories'].unique()
print(unique_categories)


['hep-ph' 'math.CO cs.CG' 'physics.gen-ph' ...
 'supr-con cond-mat.mtrl-sci cond-mat.supr-con'
 'supr-con cond-mat.mtrl-sci cond-mat.supr-con nlin.PS patt-sol'
 'supr-con cond-mat.supr-con quant-ph']


In [7]:
df['categories_list'] = df['categories'].str.split()
df

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed,categories_list
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,...",[hep-ph]
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]","[math.CO, cs.CG]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]",[physics.gen-ph]
3,0704.0004,David Callan,David Callan,A determinant of Stirling cycle numbers counts...,11 pages,None,None,None,math.CO,None,We show that a determinant of Stirling cycle...,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2007-05-23,"[[Callan, David, ]]",[math.CO]
4,0704.0005,Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,None,"Illinois J. Math. 52 (2008) no.2, 681-689",None,None,math.CA math.FA,None,In this paper we show how to compute the $\L...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2013-10-15,"[[Abu-Shammala, Wael, ], [Torchinsky, Alberto, ]]","[math.CA, math.FA]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2854534,supr-con/9608008,Ruslan Prozorov,"R. Prozorov, M. Konczykowski, B. Schmidt, Y. Y...",On the origin of the irreversibility line in t...,"19 pages, LaTex, 6 PostScript figures; Author'...",None,10.1103/PhysRevB.54.15530,None,supr-con cond-mat.supr-con,None,We report on measurements of the angular dep...,"[{'version': 'v1', 'created': 'Mon, 26 Aug 199...",2009-10-30,"[[Prozorov, R., ], [Konczykowski, M., ], [Schm...","[supr-con, cond-mat.supr-con]"
2854535,supr-con/9609001,Durga P. Choudhury,"Durga P. Choudhury, Balam A. Willemsen, John S...",Nonlinear Response of HTSC Thin Film Microwave...,"4 pages, LaTeX type, Uses IEEE style files, 60...",None,10.1109/77.620744,None,supr-con cond-mat.supr-con,None,The non-linear microwave surface impedance o...,"[{'version': 'v1', 'created': 'Sat, 31 Aug 199...",2016-11-18,"[[Choudhury, Durga P., , Physics Department, N...","[supr-con, cond-mat.supr-con]"
2854536,supr-con/9609002,Durga P. Choudhury,"Balam A. Willemsen, J. S. Derov and S.Sridhar ...",Critical State Flux Penetration and Linear Mic...,"20 pages, LaTeX type, Uses REVTeX style files,...",None,10.1103/PhysRevB.56.11989,None,supr-con cond-mat.supr-con,None,The vortex contribution to the dc field (H) ...,"[{'version': 'v1', 'created': 'Tue, 3 Sep 1996...",2009-10-30,"[[Willemsen, Balam A., , Physics Department,\n...","[supr-con, cond-mat.supr-con]"
2854537,supr-con/9609003,Hasegawa Yasumasa,Yasumasa Hasegawa (Himeji Institute of Technol...,Density of States and NMR Relaxation Rate in A...,"7 pages, 4 PostScript Figures, LaTeX, to appea...",None,10.1143/JPSJ.65.3131,None,supr-con cond-mat.supr-con,None,We show that the density of states in an ani...,"[{'version': 'v1', 'created': 'Wed, 18 Sep 199...",2009-10-30,"[[Hasegawa, Yasumasa, , Himeji Institute of Te...","[supr-con, cond-mat.supr-con]"


In [8]:
NEURAL_NETWORK_CATEGORIES = {
#type(df) TIER 1: Essential
    'cs.LG',    # Machine Learning - PRIMARY CATEGORY (90% of NN papers)
    'cs.CV',    # Computer Vision - CNNs, Vision Transformers
    'cs.CL',    # Computation and Language (NLP) - Transformers, BERT, GPT
    'cs.AI',    # Artificial Intelligence - General AI applications
    'cs.NE',    # Neural and Evolutionary Computing - Specifically NNs


# TIER 2: Important - Should include for comprehensive coverage
    # 'stat.ML',  # Statistics ML - Statistical learning theory
    # 'cs.RO',    # Robotics - RL, neural control
    # 'cs.SD',    # Sound - Audio neural networks, speech
    # 'cs.IR',    # Information Retrieval - Neural ranking, recommendations
    # 'cs.SI',    # Social and Information Networks - Graph Neural Networks
    # 'cs.CR',    # Cryptography and Security - Adversarial ML
    # 'cs.HC',    # Human-Computer Interaction - Neural interfaces
    # 'cs.MM',    # Multimedia - Video/image processing with NNs
}
def filter_categories(cat_list):
    """
    Check if:
    1. Has at least one neural network category
    2. ALL categories start with cs., math., or stat.
    """
    if not cat_list:
        return False
    
    # Check 1: Must have at least one neural network category
    has_nn_category = any(cat in NEURAL_NETWORK_CATEGORIES for cat in cat_list)
    
    # Check 2 (Optional): ALL categories must start with cs., math., or stat. 
    all_valid_prefixes = all(
        cat.startswith('cs.') or cat.startswith('math.') or cat.startswith('stat.')
        for cat in cat_list
    )
    
    return has_nn_category and all_valid_prefixes

In [11]:
mask = df['categories_list'].apply(filter_categories)
df = df[mask]
df

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed,categories_list
46,0704.0047,Igor Grabec,T. Kosel and I. Grabec,Intelligent location of simultaneously active ...,"5 pages, 5 eps figures, uses IEEEtran.cls",None,None,None,cs.NE cs.AI,None,The intelligent acoustic emission locator is...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2009-09-29,"[[Kosel, T., ], [Grabec, I., ]]","[cs.NE, cs.AI]"
49,0704.0050,Igor Grabec,T. Kosel and I. Grabec,Intelligent location of simultaneously active ...,"5 pages, 7 eps figures, uses IEEEtran.cls",None,None,None,cs.NE cs.AI,None,Part I describes an intelligent acoustic emi...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2007-05-23,"[[Kosel, T., ], [Grabec, I., ]]","[cs.NE, cs.AI]"
670,0704.0671,Maxim Raginsky,Maxim Raginsky,Learning from compressed observations,6 pages; submitted to the 2007 IEEE Informatio...,None,10.1109/ITW.2007.4313111,None,cs.IT cs.LG math.IT,None,The problem of statistical learning is to co...,"[{'version': 'v1', 'created': 'Thu, 5 Apr 2007...",2016-11-15,"[[Raginsky, Maxim, ]]","[cs.IT, cs.LG, math.IT]"
953,0704.0954,Jos\'e M. F. Moura,Soummya Kar and Jose M. F. Moura,Sensor Networks with Random Links: Topology De...,Submitted to IEEE Transactions,None,10.1109/TSP.2008.920143,None,cs.IT cs.LG math.IT,None,"In a sensor network, in practice, the commun...","[{'version': 'v1', 'created': 'Fri, 6 Apr 2007...",2009-11-13,"[[Kar, Soummya, ], [Moura, Jose M. F., ]]","[cs.IT, cs.LG, math.IT]"
984,0704.0985,Mohd Abubakr,"Mohd Abubakr, R.M.Vinay",Architecture for Pseudo Acausal Evolvable Embe...,"4 pages, 2 figures. Submitted to SASO 2007",None,None,None,cs.NE cs.AI,None,Advances in semiconductor technology are con...,"[{'version': 'v1', 'created': 'Sat, 7 Apr 2007...",2007-05-23,"[[Abubakr, Mohd, ], [Vinay, R. M., ]]","[cs.NE, cs.AI]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2783616,math/0701791,Boris Ettinger,"Boris Ettinger, Niv Sarig, Yosef Yomdin",Linear versus Non-linear Acquisition of Step-F...,"Major revision, added chapters 4, 5. 34 pages,...",None,None,None,math.CA cs.CV,None,We address in this paper the following two c...,"[{'version': 'v1', 'created': 'Sat, 27 Jan 200...",2007-11-01,"[[Ettinger, Boris, ], [Sarig, Niv, ], [Yomdin,...","[math.CA, cs.CV]"
2784571,math/0702804,Marcus Hutter,Marcus Hutter,The Loss Rank Principle for Model Selection,16 pages,Proc. 20th Annual Conf. on Learning Theory (CO...,10.1007/978-3-540-72927-3_42,None,math.ST cs.LG stat.ME stat.ML stat.TH,None,We introduce a new principle for model selec...,"[{'version': 'v1', 'created': 'Tue, 27 Feb 200...",2007-06-25,"[[Hutter, Marcus, ]]","[math.ST, cs.LG, stat.ME, stat.ML, stat.TH]"
2784633,math/0702866,Marie Cottrell,"Patrick Letr\'emy (SAMOS, CES), Marie Cottrell...",Consumer Profile Identification and Allocation,Accepted in the IWANN 07 conference San Sebast...,None,None,None,math.ST cs.NE stat.TH,None,We propose an easy-to-use methodology to all...,"[{'version': 'v1', 'created': 'Wed, 28 Feb 200...",2016-08-14,"[[Letrémy, Patrick, , SAMOS, CES], [Cottrell, ...","[math.ST, cs.NE, stat.TH]"
2787641,math/9801152,Shelah Office,"John T. Baldwin, Saharon Shelah",On the classifiability of cellular automata,None,None,None,Shelah [BlSh:623],math.LO cs.NE,None,Based on computer simulations Wolfram presen...,"[{'version': 'v1', 'created': 'Thu, 15 Jan 199...",2016-09-07,"[[Baldwin, John T., ], [Shelah, Saharon, ]]","[math.LO, cs.NE]"


### Drop the unnecessary cols

In [15]:
df = df.drop(['submitter', 'authors', 'comments', 'journal-ref', 'versions', 'license', 'report-no', 'doi', 'categories'], axis=1)
df

,id,title,abstract,update_date,authors_parsed,categories_list
46,0704.0047,Intelligent location of simultaneously active ...,The intelligent acoustic emission locator is...,2009-09-29,"[[Kosel, T., ], [Grabec, I., ]]","[cs.NE, cs.AI]"
49,0704.0050,Intelligent location of simultaneously active ...,Part I describes an intelligent acoustic emi...,2007-05-23,"[[Kosel, T., ], [Grabec, I., ]]","[cs.NE, cs.AI]"
670,0704.0671,Learning from compressed observations,The problem of statistical learning is to co...,2016-11-15,"[[Raginsky, Maxim, ]]","[cs.IT, cs.LG, math.IT]"
953,0704.0954,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun...",2009-11-13,"[[Kar, Soummya, ], [Moura, Jose M. F., ]]","[cs.IT, cs.LG, math.IT]"
984,0704.0985,Architecture for Pseudo Acausal Evolvable Embe...,Advances in semiconductor technology are con...,2007-05-23,"[[Abubakr, Mohd, ], [Vinay, R. M., ]]","[cs.NE, cs.AI]"
...,...,...,...,...,...,...
2783616,math/0701791,Linear versus Non-linear Acquisition of Step-F...,We address in this paper the following two c...,2007-11-01,"[[Ettinger, Boris, ], [Sarig, Niv, ], [Yomdin,...","[math.CA, cs.CV]"
2784571,math/0702804,The Loss Rank Principle for Model Selection,We introduce a new principle for model selec...,2007-06-25,"[[Hutter, Marcus, ]]","[math.ST, cs.LG, stat.ME, stat.ML, stat.TH]"
2784633,math/0702866,Consumer Profile Identification and Allocation,We propose an easy-to-use methodology to all...,2016-08-14,"[[Letrémy, Patrick, , SAMOS, CES], [Cottrell, ...","[math.ST, cs.NE, stat.TH]"
2787641,math/9801152,On the classifiability of cellular automata,Based on computer simulations Wolfram presen...,2016-09-07,"[[Baldwin, John T., ], [Shelah, Saharon, ]]","[math.LO, cs.NE]"


### Look for null values

In [16]:
df.isnull().sum()


id                 0
title              0
abstract           0
update_date        0
authors_parsed     0
categories_list    0
dtype: int64

#### no null/nan values left

### Rename remaining cols more appropirietly

In [17]:
df['year'] = df['update_date'].str[:4].astype(int)
df

,id,title,abstract,update_date,authors_parsed,categories_list,year
46,0704.0047,Intelligent location of simultaneously active ...,The intelligent acoustic emission locator is...,2009-09-29,"[[Kosel, T., ], [Grabec, I., ]]","[cs.NE, cs.AI]",2009
49,0704.0050,Intelligent location of simultaneously active ...,Part I describes an intelligent acoustic emi...,2007-05-23,"[[Kosel, T., ], [Grabec, I., ]]","[cs.NE, cs.AI]",2007
670,0704.0671,Learning from compressed observations,The problem of statistical learning is to co...,2016-11-15,"[[Raginsky, Maxim, ]]","[cs.IT, cs.LG, math.IT]",2016
953,0704.0954,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun...",2009-11-13,"[[Kar, Soummya, ], [Moura, Jose M. F., ]]","[cs.IT, cs.LG, math.IT]",2009
984,0704.0985,Architecture for Pseudo Acausal Evolvable Embe...,Advances in semiconductor technology are con...,2007-05-23,"[[Abubakr, Mohd, ], [Vinay, R. M., ]]","[cs.NE, cs.AI]",2007
...,...,...,...,...,...,...,...
2783616,math/0701791,Linear versus Non-linear Acquisition of Step-F...,We address in this paper the following two c...,2007-11-01,"[[Ettinger, Boris, ], [Sarig, Niv, ], [Yomdin,...","[math.CA, cs.CV]",2007
2784571,math/0702804,The Loss Rank Principle for Model Selection,We introduce a new principle for model selec...,2007-06-25,"[[Hutter, Marcus, ]]","[math.ST, cs.LG, stat.ME, stat.ML, stat.TH]",2007
2784633,math/0702866,Consumer Profile Identification and Allocation,We propose an easy-to-use methodology to all...,2016-08-14,"[[Letrémy, Patrick, , SAMOS, CES], [Cottrell, ...","[math.ST, cs.NE, stat.TH]",2016
2787641,math/9801152,On the classifiability of cellular automata,Based on computer simulations Wolfram presen...,2016-09-07,"[[Baldwin, John T., ], [Shelah, Saharon, ]]","[math.LO, cs.NE]",2016


In [18]:
df = df.drop(['update_date'], axis=1)

In [19]:
df = df.rename(columns={'id': 'arxiv_id'})
df

,arxiv_id,title,abstract,authors_parsed,categories_list,year
46,0704.0047,Intelligent location of simultaneously active ...,The intelligent acoustic emission locator is...,"[[Kosel, T., ], [Grabec, I., ]]","[cs.NE, cs.AI]",2009
49,0704.0050,Intelligent location of simultaneously active ...,Part I describes an intelligent acoustic emi...,"[[Kosel, T., ], [Grabec, I., ]]","[cs.NE, cs.AI]",2007
670,0704.0671,Learning from compressed observations,The problem of statistical learning is to co...,"[[Raginsky, Maxim, ]]","[cs.IT, cs.LG, math.IT]",2016
953,0704.0954,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun...","[[Kar, Soummya, ], [Moura, Jose M. F., ]]","[cs.IT, cs.LG, math.IT]",2009
984,0704.0985,Architecture for Pseudo Acausal Evolvable Embe...,Advances in semiconductor technology are con...,"[[Abubakr, Mohd, ], [Vinay, R. M., ]]","[cs.NE, cs.AI]",2007
...,...,...,...,...,...,...
2783616,math/0701791,Linear versus Non-linear Acquisition of Step-F...,We address in this paper the following two c...,"[[Ettinger, Boris, ], [Sarig, Niv, ], [Yomdin,...","[math.CA, cs.CV]",2007
2784571,math/0702804,The Loss Rank Principle for Model Selection,We introduce a new principle for model selec...,"[[Hutter, Marcus, ]]","[math.ST, cs.LG, stat.ME, stat.ML, stat.TH]",2007
2784633,math/0702866,Consumer Profile Identification and Allocation,We propose an easy-to-use methodology to all...,"[[Letrémy, Patrick, , SAMOS, CES], [Cottrell, ...","[math.ST, cs.NE, stat.TH]",2016
2787641,math/9801152,On the classifiability of cellular automata,Based on computer simulations Wolfram presen...,"[[Baldwin, John T., ], [Shelah, Saharon, ]]","[math.LO, cs.NE]",2016


In [20]:
df['article_url'] = df['arxiv_id'].apply(lambda x: f"https://arxiv.org/pdf/{x}.pdf")
df

,arxiv_id,title,abstract,authors_parsed,categories_list,year,article_url
46,0704.0047,Intelligent location of simultaneously active ...,The intelligent acoustic emission locator is...,"[[Kosel, T., ], [Grabec, I., ]]","[cs.NE, cs.AI]",2009,https://arxiv.org/pdf/0704.0047.pdf
49,0704.0050,Intelligent location of simultaneously active ...,Part I describes an intelligent acoustic emi...,"[[Kosel, T., ], [Grabec, I., ]]","[cs.NE, cs.AI]",2007,https://arxiv.org/pdf/0704.0050.pdf
670,0704.0671,Learning from compressed observations,The problem of statistical learning is to co...,"[[Raginsky, Maxim, ]]","[cs.IT, cs.LG, math.IT]",2016,https://arxiv.org/pdf/0704.0671.pdf
953,0704.0954,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun...","[[Kar, Soummya, ], [Moura, Jose M. F., ]]","[cs.IT, cs.LG, math.IT]",2009,https://arxiv.org/pdf/0704.0954.pdf
984,0704.0985,Architecture for Pseudo Acausal Evolvable Embe...,Advances in semiconductor technology are con...,"[[Abubakr, Mohd, ], [Vinay, R. M., ]]","[cs.NE, cs.AI]",2007,https://arxiv.org/pdf/0704.0985.pdf
...,...,...,...,...,...,...,...
2783616,math/0701791,Linear versus Non-linear Acquisition of Step-F...,We address in this paper the following two c...,"[[Ettinger, Boris, ], [Sarig, Niv, ], [Yomdin,...","[math.CA, cs.CV]",2007,https://arxiv.org/pdf/math/0701791.pdf
2784571,math/0702804,The Loss Rank Principle for Model Selection,We introduce a new principle for model selec...,"[[Hutter, Marcus, ]]","[math.ST, cs.LG, stat.ME, stat.ML, stat.TH]",2007,https://arxiv.org/pdf/math/0702804.pdf
2784633,math/0702866,Consumer Profile Identification and Allocation,We propose an easy-to-use methodology to all...,"[[Letrémy, Patrick, , SAMOS, CES], [Cottrell, ...","[math.ST, cs.NE, stat.TH]",2016,https://arxiv.org/pdf/math/0702866.pdf
2787641,math/9801152,On the classifiability of cellular automata,Based on computer simulations Wolfram presen...,"[[Baldwin, John T., ], [Shelah, Saharon, ]]","[math.LO, cs.NE]",2016,https://arxiv.org/pdf/math/9801152.pdf


#### filter out articles prior to 2012 (before deep learning era)

In [21]:
df = df[df['year'] >= 2012].copy()
df

,arxiv_id,title,abstract,authors_parsed,categories_list,year,article_url
670,0704.0671,Learning from compressed observations,The problem of statistical learning is to co...,"[[Raginsky, Maxim, ]]","[cs.IT, cs.LG, math.IT]",2016,https://arxiv.org/pdf/0704.0671.pdf
1408,0704.1409,Preconditioned Temporal Difference Learning,This paper has been withdrawn by the author....,"[[HengShuai, Yao, ]]","[cs.LG, cs.AI]",2012,https://arxiv.org/pdf/0704.1409.pdf
1674,0704.1675,Exploiting Social Annotation for Automatic Res...,"Information integration applications, such a...","[[Plangprasopchok, Anon, ], [Lerman, Kristina, ]]","[cs.AI, cs.CY, cs.DL]",2016,https://arxiv.org/pdf/0704.1675.pdf
3514,0704.3515,Comparing Robustness of Pairwise and Multiclas...,"Noise, corruptions and variations in face im...","[[Uglov, J., ], [Schetinin, V., ], [Maple, C., ]]",[cs.AI],2016,https://arxiv.org/pdf/0704.3515.pdf
3661,0704.3662,An Automated Evaluation Metric for Chinese Tex...,"In this paper, we propose an automated evalu...","[[Jiang, Mike Tian-Jian, ], [Zhan, James, ], [...","[cs.HC, cs.CL]",2013,https://arxiv.org/pdf/0704.3662.pdf
...,...,...,...,...,...,...,...
2771271,math/0510276,An algorithmic and a geometric characterizatio...,We show that the class of conditional distri...,"[[Gill, Richard D., , Leiden University], [Gru...","[math.ST, cs.AI, stat.ME, stat.TH]",2023,https://arxiv.org/pdf/math/0510276.pdf
2781435,math/0611433,Working times in atypical forms of employment:...,"In the present article, we attempt to devise...","[[Letrémy, Patrick, , SAMOS], [Cottrell, Marie...","[math.ST, cs.NE, stat.TH]",2016,https://arxiv.org/pdf/math/0611433.pdf
2782966,math/0701142,On the use of self-organizing maps to accelera...,Self-organizing maps (SOM) are widely used f...,"[[De Bodt, Eric, , ESA, Iag-Fin, Dice], [Cottr...","[math.ST, cs.NE, stat.TH]",2016,https://arxiv.org/pdf/math/0701142.pdf
2784633,math/0702866,Consumer Profile Identification and Allocation,We propose an easy-to-use methodology to all...,"[[Letrémy, Patrick, , SAMOS, CES], [Cottrell, ...","[math.ST, cs.NE, stat.TH]",2016,https://arxiv.org/pdf/math/0702866.pdf


In [22]:
from datasets import load_dataset

# 1. Load the dataset from Hugging Face
# Specify the dataset ID and the configuration (usually 'default')
dataset_id = "pwc-archive/links-between-paper-and-code"

print(f"Loading dataset: {dataset_id}")
dataset = load_dataset(dataset_id)
print("Dataset loaded successfully!")

Loading dataset: pwc-archive/links-between-paper-and-code


README.md:   0%|          | 0.00/826 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300161 [00:00<?, ? examples/s]

Dataset loaded successfully!


In [26]:
df_links = dataset['train'].to_pandas()
df_links

,paper_url,paper_title,paper_arxiv_id,paper_url_abs,paper_url_pdf,repo_url,is_official,mentioned_in_paper,mentioned_in_github,framework
0,https://paperswithcode.com/paper/odyssey-a-pub...,Odyssey: A Public GPU-Based Code for General-R...,1601.02063,https://arxiv.org/abs/1601.02063v2,https://arxiv.org/pdf/1601.02063v2.pdf,https://github.com/LeonGeiger/Kerr,False,False,True,none
1,https://paperswithcode.com/paper/efficient-lea...,Efficient leave-one-out cross-validation for B...,1810.10559,https://arxiv.org/abs/1810.10559v5,https://arxiv.org/pdf/1810.10559v5.pdf,https://github.com/paul-buerkner/psis-non-fact...,True,True,False,none
2,https://paperswithcode.com/paper/automatic-pos...,Automatic Post-Editing of Machine Translation:...,None,https://aclanthology.org/D18-1341,https://aclanthology.org/D18-1341.pdf,https://github.com/trangvu/ape-npi,False,False,False,tf
3,https://paperswithcode.com/paper/attngan-fine-...,AttnGAN: Fine-Grained Text to Image Generation...,1711.10485,http://arxiv.org/abs/1711.10485v1,http://arxiv.org/pdf/1711.10485v1.pdf,https://github.com/bprabhakar/text-to-image,False,False,False,pytorch
4,https://paperswithcode.com/paper/photo-realist...,Photo-Realistic Single Image Super-Resolution ...,1609.04802,http://arxiv.org/abs/1609.04802v5,http://arxiv.org/pdf/1609.04802v5.pdf,https://github.com/2023-MindSpore-1/ms-code-21...,False,False,False,mindspore
...,...,...,...,...,...,...,...,...,...,...
300156,https://paperswithcode.com/paper/plsrglm-parti...,plsRglm: Partial least squares linear and gene...,1810.01005,http://arxiv.org/abs/1810.01005v1,http://arxiv.org/pdf/1810.01005v1.pdf,https://github.com/fbertran/plsRglm,False,False,True,none
300157,https://paperswithcode.com/paper/a-multilayer-...,A Multilayer Convolutional Encoder-Decoder Neu...,1801.08831,http://arxiv.org/abs/1801.08831v1,http://arxiv.org/pdf/1801.08831v1.pdf,https://github.com/seaweiqing/neuraltalk_plus_...,False,False,True,tf
300158,https://paperswithcode.com/paper/unsupervised-...,Unsupervised domain adaptation for medical ima...,1811.06042,http://arxiv.org/abs/1811.06042v2,http://arxiv.org/pdf/1811.06042v2.pdf,https://github.com/neuropoly/domainadaptation,True,True,False,pytorch
300159,https://paperswithcode.com/paper/overlapping-c...,Overlapping Community Detection at Scale: A No...,None,https://dl.acm.org/citation.cfm?id=2433471,http://infolab.stanford.edu/~crucis/pubs/paper...,https://github.com/benedekrozemberczki/karateclub,False,False,False,none


In [ ]:
df_links = df_links.drop(['paper_title', 'paper_url', 'paper_url_pdf',  'is_official', 'mentioned_in_paper', 'mentioned_in_github',	'framework', 'paper_url_abs' ] , axis=1)
df_links

In [48]:
df_links.isna().sum()

paper_arxiv_id    0
repo_url          0
dtype: int64

In [49]:
df_links = df_links.dropna()
df_links

,paper_arxiv_id,repo_url
0,1601.02063,https://github.com/LeonGeiger/Kerr
1,1810.10559,https://github.com/paul-buerkner/psis-non-fact...
3,1711.10485,https://github.com/bprabhakar/text-to-image
4,1609.04802,https://github.com/2023-MindSpore-1/ms-code-21...
5,2101.08393,https://github.com/google/pwlfit
...,...,...
300155,1505.07570,https://github.com/wangshusen/RandMatrixMatlab
300156,1810.01005,https://github.com/fbertran/plsRglm
300157,1801.08831,https://github.com/seaweiqing/neuraltalk_plus_...
300158,1811.06042,https://github.com/neuropoly/domainadaptation


In [50]:
df_papers = df.copy()
df_paper_code = df_links.copy()

### Aggregate by paper_id and store url's as lsit

In [51]:
df_paper_code = df_paper_code.groupby('paper_arxiv_id', as_index=False).agg({
    'repo_url': list,       # Collects all GitHub/repository URLs into a list
    'paper_arxiv_id': 'first' # Keep the first occurrence of the ID 
})
df_paper_code

,repo_url,paper_arxiv_id
0,[https://github.com/miku/xmlcutty],0704.0004
1,[https://github.com/miku/xmlcutty],0704.0010
2,[https://github.com/miku/xmlcutty],0704.0012
3,[https://github.com/adda-team/adda],0704.0033
4,[https://github.com/adda-team/adda],0704.0035
...,...,...
207410,[https://github.com/DeBueno/Qsharp-Quantum-Sec...,quant-ph/9806063
207411,[https://github.com/iQuHACK/2021_Pauli-Wanna-C...,quant-ph/9806088
207412,[https://github.com/nrenga/ghz_distillation_qe...,quant-ph/9807006
207413,"[https://github.com/hhy37/Liquid, https://gith...",quant-ph/9807053


In [52]:
merged_df = pd.merge(
    left=df_papers,
    right=df_paper_code,
    left_on='arxiv_id',  # Column name in df1
    right_on='paper_arxiv_id', # Column name in df2
    how='inner'
)

In [53]:
merged_df

,arxiv_id,title,abstract,authors_parsed,categories_list,year,article_url,repo_url,paper_arxiv_id
0,0705.4676,Recursive n-gram hashing is pairwise independe...,Many applications use sequences of n consecu...,"[[Lemire, Daniel, ], [Kaser, Owen, ]]","[cs.DB, cs.CL]",2016,https://arxiv.org/pdf/0705.4676.pdf,"[https://github.com/zhaoxiaofei/bindash, https...",0705.4676
1,0804.4451,Dependence Structure Estimation via Copula,Dependence strucuture estimation is one of t...,"[[Ma, Jian, ], [Sun, Zengqi, ]]","[cs.LG, cs.IR, stat.ME]",2019,https://arxiv.org/pdf/0804.4451.pdf,[https://github.com/majianthu/dse],0804.4451
2,0811.3301,Faster Retrieval with a Two-Pass Dynamic-Time-...,The Dynamic Time Warping (DTW) is a popular ...,"[[Lemire, Daniel, ]]","[cs.DB, cs.CV]",2012,https://arxiv.org/pdf/0811.3301.pdf,[https://github.com/lemire/lbimproved],0811.3301
3,0902.4682,Lectures on Jacques Herbrand as a Logician,We give some lectures on the work on formal ...,"[[Wirth, Claus-Peter, ], [Siekmann, Joerg, ], ...","[cs.LO, cs.AI]",2014,https://arxiv.org/pdf/0902.4682.pdf,[https://github.com/thejohncrafter/flows],0902.4682
4,0906.2027,Matrix Completion from Noisy Entries,"Given a matrix M of low-rank, we consider th...","[[Keshavan, Raghunandan H., ], [Montanari, And...","[cs.LG, stat.ML]",2012,https://arxiv.org/pdf/0906.2027.pdf,[https://github.com/jasonsun0310/MatrixComplet...,0906.2027
...,...,...,...,...,...,...,...,...,...
127268,2507.15351,One Step is Enough: Multi-Agent Reinforcement ...,On-demand ride-sharing platforms face the fund...,"[[Zhao, Zijian, ], [Li, Sen, ]]","[cs.AI, cs.ET, cs.MA]",2025,https://arxiv.org/pdf/2507.15351.pdf,[https://github.com/RS2002/OSPO],2507.15351
127269,2507.15454,ObjectGS: Object-aware Scene Reconstruction an...,3D Gaussian Splatting is renowned for its high...,"[[Zhu, Ruijie, ], [Yu, Mulin, ], [Xu, Linning,...","[cs.GR, cs.AI, cs.CV, cs.HC]",2025,https://arxiv.org/pdf/2507.15454.pdf,[https://github.com/RuijieZhu94/ObjectGS],2507.15454
127270,2507.15641,Leveraging Context for Multimodal Fallacy Clas...,"In this paper, we present our submission to th...","[[Pittiglio, Alessio, ]]","[cs.CL, cs.AI]",2025,https://arxiv.org/pdf/2507.15641.pdf,[https://github.com/alessiopittiglio/mm-argfal...,2507.15641
127271,cs/0212008,Principal Manifolds and Nonlinear Dimension Re...,Nonlinear manifold learning from unorganized...,"[[Zhang, Zhenyue, ], [Zha, Hongyuan, ]]","[cs.LG, cs.AI]",2016,https://arxiv.org/pdf/cs/0212008.pdf,[https://github.com/gitr00ki3/vpw],cs/0212008


In [55]:
merged_df['arxiv_id'].value_counts()

arxiv_id
0705.4676     1
2310.08755    1
2310.08889    1
2310.08887    1
2310.08885    1
             ..
2106.06682    1
2106.06672    1
2106.06667    1
2106.06666    1
cs/0702144    1
Name: count, Length: 127273, dtype: int64

In [56]:
merged_df = merged_df.drop(['paper_arxiv_id' ] , axis=1)
merged_df

,arxiv_id,title,abstract,authors_parsed,categories_list,year,article_url,repo_url
0,0705.4676,Recursive n-gram hashing is pairwise independe...,Many applications use sequences of n consecu...,"[[Lemire, Daniel, ], [Kaser, Owen, ]]","[cs.DB, cs.CL]",2016,https://arxiv.org/pdf/0705.4676.pdf,"[https://github.com/zhaoxiaofei/bindash, https..."
1,0804.4451,Dependence Structure Estimation via Copula,Dependence strucuture estimation is one of t...,"[[Ma, Jian, ], [Sun, Zengqi, ]]","[cs.LG, cs.IR, stat.ME]",2019,https://arxiv.org/pdf/0804.4451.pdf,[https://github.com/majianthu/dse]
2,0811.3301,Faster Retrieval with a Two-Pass Dynamic-Time-...,The Dynamic Time Warping (DTW) is a popular ...,"[[Lemire, Daniel, ]]","[cs.DB, cs.CV]",2012,https://arxiv.org/pdf/0811.3301.pdf,[https://github.com/lemire/lbimproved]
3,0902.4682,Lectures on Jacques Herbrand as a Logician,We give some lectures on the work on formal ...,"[[Wirth, Claus-Peter, ], [Siekmann, Joerg, ], ...","[cs.LO, cs.AI]",2014,https://arxiv.org/pdf/0902.4682.pdf,[https://github.com/thejohncrafter/flows]
4,0906.2027,Matrix Completion from Noisy Entries,"Given a matrix M of low-rank, we consider th...","[[Keshavan, Raghunandan H., ], [Montanari, And...","[cs.LG, stat.ML]",2012,https://arxiv.org/pdf/0906.2027.pdf,[https://github.com/jasonsun0310/MatrixComplet...
...,...,...,...,...,...,...,...,...
127268,2507.15351,One Step is Enough: Multi-Agent Reinforcement ...,On-demand ride-sharing platforms face the fund...,"[[Zhao, Zijian, ], [Li, Sen, ]]","[cs.AI, cs.ET, cs.MA]",2025,https://arxiv.org/pdf/2507.15351.pdf,[https://github.com/RS2002/OSPO]
127269,2507.15454,ObjectGS: Object-aware Scene Reconstruction an...,3D Gaussian Splatting is renowned for its high...,"[[Zhu, Ruijie, ], [Yu, Mulin, ], [Xu, Linning,...","[cs.GR, cs.AI, cs.CV, cs.HC]",2025,https://arxiv.org/pdf/2507.15454.pdf,[https://github.com/RuijieZhu94/ObjectGS]
127270,2507.15641,Leveraging Context for Multimodal Fallacy Clas...,"In this paper, we present our submission to th...","[[Pittiglio, Alessio, ]]","[cs.CL, cs.AI]",2025,https://arxiv.org/pdf/2507.15641.pdf,[https://github.com/alessiopittiglio/mm-argfal...
127271,cs/0212008,Principal Manifolds and Nonlinear Dimension Re...,Nonlinear manifold learning from unorganized...,"[[Zhang, Zhenyue, ], [Zha, Hongyuan, ]]","[cs.LG, cs.AI]",2016,https://arxiv.org/pdf/cs/0212008.pdf,[https://github.com/gitr00ki3/vpw]


### Save merged dataframe to parquet to preserve url list structure

In [58]:
output_filename = 'neural_net_papers_metadata.parquet.gzip'

merged_df.to_parquet(
    output_filename,
    index=False,
    compression='gzip'  
)

print(f"Saved safely to Parquet: {output_filename}")

Saved safely to Parquet: neural_net_papers_metadata.parquet.gzip
